In [20]:
import essentia.standard as es
import numpy as np
import subprocess
import numpy as np
from PIL import Image
import os

In [14]:
def load_audio_as_spectrogram_essentia(file_path):
    """
    Loads an audio file and computes its log-magnitude spectrogram using Essentia.
    """
    # 1. Load the audio file 
    # MonoLoader automatically downmixes multi-channel files to mono 
    # and resamples to a uniform 44100Hz by default.
    audio = es.MonoLoader(filename=file_path)()
    
    # 2. Instantiate the required spectral algorithms
    windowing = es.Windowing(type='hann')
    spectrum = es.Spectrum()
    
    # 3. Slice the audio into frames and compute the linear magnitude spectrum
    # frameSize=2048 and hopSize=512 perfectly match standard STFT dimensions
    spec_list = []
    for frame in es.FrameGenerator(audio, frameSize=2048, hopSize=512):
        windowed_frame = windowing(frame)
        frame_spectrum = spectrum(windowed_frame)
        spec_list.append(frame_spectrum)
        
    # 4. Stack frames into a 2D NumPy array (Frequency Bins x Time Frames)
    spectrogram = np.array(spec_list).T
    
    # 5. Convert linear amplitude to Decibel (Log) scale
    # We use np.maximum to clip values at 1e-5 to avoid log(0) baseline crashes
    spectrogram_db = 20 * np.log10(np.maximum(spectrogram, 1e-5))
    
    return spectrogram_db

In [12]:
def compute_ck1_distance(spec_x, spec_y, target_size=(256, 256), quality=5):
    """
    Computes the Campana-Keogh (CK-1) distance between two 2D spectrogram arrays.
    
    Parameters:
    -----------
    spec_x : np.ndarray
        2D array of the first spectrogram.
    spec_y : np.ndarray
        2D array of the second spectrogram.
    target_size : tuple (width, height)
        Dimensions to resize spectrograms. Must be multiples of 16 for MPEG-1 (e.g., 256x256).
    quality : int
        MPEG fixed quality scale (1-31). Lower means higher quality/finer detail resolution.
        Default 5 is a robust sweet spot for texture discovery.
    """
    
    # 1. Helper function to normalize and resize spectrograms to grayscale frames
    def preprocess_spectrogram(spec):
        # Normalize strictly to 0-255 grayscale range
        s_min, s_max = spec.min(), spec.max()
        if s_max > s_min:
            spec_norm = 255.0 * (spec - s_min) / (s_max - s_min)
        else:
            spec_norm = np.zeros_like(spec)
            
        # Resize to standard uniform dimensions using Pillow
        img = Image.fromarray(spec_norm.astype(np.uint8))
        img_resized = img.resize(target_size, Image.Resampling.BILINEAR)
        return np.array(img_resized)

    # Preprocess both inputs
    x = preprocess_spectrogram(spec_x)
    y = preprocess_spectrogram(spec_y)
    width, height = target_size

    # 2. Helper function to pass 2 frames to FFmpeg and get the compressed size in bytes
    def get_mpeg1_compressed_size(frame_1, frame_2):
        f1_bytes = frame_1.tobytes()
        f2_bytes = frame_2.tobytes()
        
        # FFmpeg command optimized for exact conditional algorithmic complexity estimation
        cmd = [
            'ffmpeg', '-y',
            '-f', 'rawvideo',
            '-pix_fmt', 'gray',
            '-s', f'{width}x{height}',
            '-r', '25',                    # Standard framerate input
            '-i', 'pipe:0',                # Read from stdin pipe
            '-vcodec', 'mpeg1video',       # Force legacy MPEG-1 as per the original spec
            '-bf', '0',                    # Disable B-frames (forces frame 2 to be a P-frame)
            '-g', '10',                    # Prevent forcing frame 2 into a new GOP/I-frame
            '-q:v', str(quality),          # CRITICAL: Use constant quality scale instead of fixed bitrate
            '-f', 'mpeg1video',            # Raw video stream container (minimal overhead)
            'pipe:1'                       # Output to stdout pipe
        ]
        
        # Open asynchronous pipe to FFmpeg
        process = subprocess.Popen(
            cmd, stdin=subprocess.PIPE, stdout=subprocess.PIPE, stderr=subprocess.PIPE
        )
        
        # Stream frames into stdin and grab output bitstream from stdout
        stdout, _ = process.communicate(input=f1_bytes + f2_bytes)
        return len(stdout)

    # 3. Compute the four compression elements required by the formula
    c_x_given_y = get_mpeg1_compressed_size(y, x)  # y is I-frame, x is P-frame
    c_y_given_x = get_mpeg1_compressed_size(x, y)  # x is I-frame, y is P-frame
    c_x_given_x = get_mpeg1_compressed_size(x, x)  # Identity baseline x
    c_y_given_y = get_mpeg1_compressed_size(y, y)  # Identity baseline y

    # 4. Final CK-1 Metric calculation
    numerator = c_x_given_y + c_y_given_x
    denominator = c_x_given_x + c_y_given_y
    
    ck1_distance = (numerator / denominator) - 1.0
    return max(0.0, ck1_distance) # Clamp near zero minor float variations

In [36]:
!ls ../data/audio/pickaxe

240801_2698284-hq.ogg  588306_12594692-hq.ogg  674381_13732472-hq.ogg
362711_6460155-hq.ogg  635702_5685306-hq.ogg


In [23]:
spec1 = load_audio_as_spectrogram_essentia("../data/audio/pickaxe/674381_13732472-hq.ogg")

In [24]:
spec1.shape

(1025, 449)

In [38]:
spec2 = load_audio_as_spectrogram_essentia("../data/audio/pickaxe/240801_2698284-hq.ogg")

In [39]:
distance = compute_ck1_distance(spec1, spec2)

In [40]:
distance

0.6583238134801122

In [33]:
spec3 = load_audio_as_spectrogram_essentia("../data/audio/rain/116961_2081635-hq.ogg")

In [34]:
distance = compute_ck1_distance(spec1, spec3)

In [35]:
distance

0.482476557824034